## Setup

This notebook performs exploratory data analysis on the four datasets used in the study:

1. **Neurodegenerative**: Chilean Spanish, clinical groups (PD, bvFTD, HC)
2. **Swear Fluency**: U.S. English, verbal fluency task
3. **Italian**: Property listing task
4. **German**: Property listing task

In [ ]:
import os
import sys
import importlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

sys.path.append("../src")

import mappings
import plot

# Reload modules to apply any changes
importlib.reload(mappings)
importlib.reload(plot)

%matplotlib inline

In [ ]:
# Dataset files
datasets = {
    "Neurodegenerative": "parkinson",
    "Swear Fluency": "swear-fluency",
    "Italian": "italian",
    "German": "german",
}

# Load all datasets
dfs = {}
for name, file in datasets.items():
    dfs[name] = pd.read_csv(f"../data/raw/{file}.csv")
    print(f"{name}: {dfs[name].shape}")

## 1. Neurodegenerative Dataset

Chilean Spanish-speaking participants with Parkinson's disease (PD), behavioral variant frontotemporal dementia (bvFTD), and healthy controls (HC).

In [ ]:
df_neuro = dfs["Neurodegenerative"]
df_neuro.head()

### 1.1 Basic Statistics

In [ ]:
# Participant and trial counts
n_participants = df_neuro["id"].nunique()
n_concepts = df_neuro["concept"].nunique()
n_properties = len(df_neuro)

print(f"Number of participants: {n_participants}")
print(f"Number of concepts: {n_concepts}")
print(f"Total properties generated: {n_properties}")
print(f"\nConcepts: {sorted(df_neuro['concept'].unique())}")

In [ ]:
# Category distribution
print("Category distribution:")
category_dist = df_neuro["category"].value_counts()
print(category_dist)

print(f"\nParticipants per category:")
for cat in df_neuro["category"].unique():
    n = df_neuro[df_neuro["category"] == cat]["id"].nunique()
    print(f"  {cat}: {n} participants")

In [ ]:
# Trajectory lengths (properties per trial)
trajectory_lengths = df_neuro.groupby(["id", "concept"]).size()

# Plot distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall distribution
axes[0].hist(trajectory_lengths, bins=30, edgecolor="black", alpha=0.7)
axes[0].axvline(
    trajectory_lengths.mean(),
    color="red",
    linestyle="--",
    label=f"Mean: {trajectory_lengths.mean():.1f}",
)
axes[0].axvline(
    trajectory_lengths.median(),
    color="orange",
    linestyle="--",
    label=f"Median: {trajectory_lengths.median():.1f}",
)
axes[0].set_xlabel("Number of Properties per Trial")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Distribution of Trajectory Lengths")
axes[0].legend()
axes[0].grid(alpha=0.3)

# By category
for cat in df_neuro["category"].unique():
    cat_df = df_neuro[df_neuro["category"] == cat]
    cat_lengths = cat_df.groupby(["id", "concept"]).size()
    axes[1].hist(
        cat_lengths,
        bins=20,
        alpha=0.5,
        label=mappings.categories["parkinson"].get(cat, cat),
    )

axes[1].set_xlabel("Number of Properties per Trial")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Trajectory Lengths by Clinical Group")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Trajectory length statistics:")
print(trajectory_lengths.describe())

### 1.2 Text-Level Exploration

In [ ]:
# Most common properties
property_counts = df_neuro["property"].value_counts().head(20)

# Visualize
plt.figure(figsize=(12, 6))
property_counts.plot(kind="barh")
plt.xlabel("Frequency")
plt.ylabel("Property")
plt.title("Top 20 Most Common Properties")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("Top 20 most common properties:")
print(property_counts)

In [ ]:
# Property diversity
total_properties = len(df_neuro)
unique_properties = df_neuro["property"].nunique()
repetition_rate = 1 - (unique_properties / total_properties)

print(f"Total properties: {total_properties}")
print(f"Unique properties: {unique_properties}")
print(f"Repetition rate: {repetition_rate:.2%}")

# Property word length
df_neuro["word_length"] = df_neuro["property"].str.len()
df_neuro["word_count"] = df_neuro["property"].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_neuro["word_length"].dropna(), bins=30, edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Character Length")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Distribution of Property Character Lengths")
axes[0].grid(alpha=0.3)

axes[1].hist(df_neuro["word_count"].dropna(), bins=20, edgecolor="black", alpha=0.7)
axes[1].set_xlabel("Word Count")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Distribution of Property Word Counts")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 1.3 Category Comparisons

In [ ]:
# Trajectory lengths by category
trajectory_by_cat = (
    df_neuro.groupby(["category", "id", "concept"])
    .size()
    .reset_index(name="n_properties")
)

plt.figure(figsize=(10, 6))
sns.boxplot(
    data=trajectory_by_cat, x="category", y="n_properties", order=["CN", "PD", "DF"]
)
plt.xlabel("Clinical Group")
plt.ylabel("Number of Properties per Trial")
plt.title("Trajectory Lengths by Clinical Group")
plt.xticks([0, 1, 2], ["HC", "PD", "bvFTD"])
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\nMean trajectory lengths by category:")
print(trajectory_by_cat.groupby("category")["n_properties"].mean())

In [ ]:
# Vocabulary richness by category (unique properties per participant)
vocab_richness = (
    df_neuro.groupby(["category", "id"])["property"]
    .nunique()
    .reset_index(name="unique_properties")
)

plt.figure(figsize=(10, 6))
sns.boxplot(
    data=vocab_richness, x="category", y="unique_properties", order=["CN", "PD", "DF"]
)
plt.xlabel("Clinical Group")
plt.ylabel("Unique Properties per Participant")
plt.title("Vocabulary Richness by Clinical Group")
plt.xticks([0, 1, 2], ["HC", "PD", "bvFTD"])
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\nMean unique properties by category:")
print(vocab_richness.groupby("category")["unique_properties"].mean())

In [ ]:
# Top properties by category
print("\nTop 10 properties by clinical group:\n")
for cat in ["CN", "DF", "PD"]:
    cat_label = mappings.categories["parkinson"][cat]
    print(f"\n{cat_label}:")
    top_props = (
        df_neuro[df_neuro["category"] == cat]["property"].value_counts().head(10)
    )
    print(top_props)

## 2. Swear Fluency Dataset

U.S. English speakers performing verbal fluency tasks across different categories including swear words.

In [ ]:
df_swear = dfs["Swear Fluency"]
df_swear.head()

### 2.1 Basic Statistics

In [ ]:
# Participant and trial counts
n_participants = df_swear["id"].nunique()
n_concepts = df_swear["concept"].nunique()
n_properties = len(df_swear)

print(f"Number of participants: {n_participants}")
print(f"Number of categories: {n_concepts}")
print(f"Total words generated: {n_properties}")
print(f"\nCategories: {sorted(df_swear['concept'].unique())}")

In [ ]:
# Category distribution
category_dist = df_swear["concept"].value_counts()

plt.figure(figsize=(10, 6))
category_dist.plot(kind="bar", edgecolor="black", alpha=0.7)
plt.xlabel("Category")
plt.ylabel("Number of Words")
plt.title("Word Count by Category")
plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Category distribution:")
print(category_dist)

In [ ]:
# Trajectory lengths (words per trial)
trajectory_lengths = df_swear.groupby(["id", "concept"]).size()

# Plot distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall distribution
axes[0].hist(trajectory_lengths, bins=30, edgecolor="black", alpha=0.7)
axes[0].axvline(
    trajectory_lengths.mean(),
    color="red",
    linestyle="--",
    label=f"Mean: {trajectory_lengths.mean():.1f}",
)
axes[0].axvline(
    trajectory_lengths.median(),
    color="orange",
    linestyle="--",
    label=f"Median: {trajectory_lengths.median():.1f}",
)
axes[0].set_xlabel("Number of Words per Trial")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Distribution of Trajectory Lengths")
axes[0].legend()
axes[0].grid(alpha=0.3)

# By category
for cat in df_swear["concept"].unique():
    cat_df = df_swear[df_swear["concept"] == cat]
    cat_lengths = cat_df.groupby("id").size()
    axes[1].hist(
        cat_lengths,
        bins=20,
        alpha=0.5,
        label=mappings.categories["swear-fluency"].get(cat, cat),
    )

axes[1].set_xlabel("Number of Words per Trial")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Trajectory Lengths by Category")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Trajectory length statistics:")
print(trajectory_lengths.describe())

### 2.2 Text-Level Exploration

In [ ]:
# Most common words
word_counts = df_swear["property"].value_counts().head(20)

# Visualize
plt.figure(figsize=(12, 6))
word_counts.plot(kind="barh")
plt.xlabel("Frequency")
plt.ylabel("Word")
plt.title("Top 20 Most Common Words")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("Top 20 most common words:")
print(word_counts)

In [ ]:
# Word diversity
total_words = len(df_swear)
unique_words = df_swear["property"].nunique()
repetition_rate = 1 - (unique_words / total_words)

print(f"Total words: {total_words}")
print(f"Unique words: {unique_words}")
print(f"Repetition rate: {repetition_rate:.2%}")

# Word length
df_swear["word_length"] = df_swear["property"].str.len()

plt.figure(figsize=(10, 6))
plt.hist(df_swear["word_length"].dropna(), bins=30, edgecolor="black", alpha=0.7)
plt.xlabel("Character Length")
plt.ylabel("Frequency")
plt.title("Distribution of Word Character Lengths")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 2.3 Category Comparisons

In [ ]:
# Trajectory lengths by category
trajectory_by_cat = (
    df_swear.groupby(["concept", "id"]).size().reset_index(name="n_words")
)

plt.figure(figsize=(12, 6))
sns.boxplot(
    data=trajectory_by_cat,
    x="concept",
    y="n_words",
    order=["ANIMAL", "A_LETTER", "F_LETTER", "S_LETTER", "SWEAR_WORDS"],
)
plt.xlabel("Category")
plt.ylabel("Number of Words per Trial")
plt.title("Trajectory Lengths by Category")
labels = [
    mappings.categories["swear-fluency"][cat]
    for cat in ["ANIMAL", "A_LETTER", "F_LETTER", "S_LETTER", "SWEAR_WORDS"]
]
plt.xticks(range(5), labels, rotation=45)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\nMean trajectory lengths by category:")
print(
    trajectory_by_cat.groupby("concept")["n_words"].mean().sort_values(ascending=False)
)

In [ ]:
# Vocabulary richness by category (unique words per participant)
vocab_richness = (
    df_swear.groupby(["concept", "id"])["property"]
    .nunique()
    .reset_index(name="unique_words")
)

plt.figure(figsize=(12, 6))
sns.boxplot(
    data=vocab_richness,
    x="concept",
    y="unique_words",
    order=["ANIMAL", "A_LETTER", "F_LETTER", "S_LETTER", "SWEAR_WORDS"],
)
plt.xlabel("Category")
plt.ylabel("Unique Words per Participant")
plt.title("Vocabulary Richness by Category")
labels = [
    mappings.categories["swear-fluency"][cat]
    for cat in ["ANIMAL", "A_LETTER", "F_LETTER", "S_LETTER", "SWEAR_WORDS"]
]
plt.xticks(range(5), labels, rotation=45)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\nMean unique words by category:")
print(
    vocab_richness.groupby("concept")["unique_words"]
    .mean()
    .sort_values(ascending=False)
)

In [ ]:
# Top words by category
print("\nTop 10 words by category:\n")
for cat in ["ANIMAL", "A_LETTER", "F_LETTER", "S_LETTER", "SWEAR_WORDS"]:
    cat_label = mappings.categories["swear-fluency"][cat]
    print(f"\n{cat_label}:")
    top_words = df_swear[df_swear["concept"] == cat]["property"].value_counts().head(10)
    print(top_words)

## 3. Italian Dataset

Italian students generating descriptive properties for 50 concrete concepts across 10 categories.

In [ ]:
df_italian = dfs["Italian"]
df_italian.head()

### 3.1 Basic Statistics

In [ ]:
# Participant and trial counts
n_participants = df_italian["id"].nunique()
n_concepts = df_italian["concept"].nunique()
n_categories = df_italian["category"].nunique()
n_properties = len(df_italian)

print(f"Number of participants: {n_participants}")
print(f"Number of concepts: {n_concepts}")
print(f"Number of categories: {n_categories}")
print(f"Total properties generated: {n_properties}")
print(f"\nCategories: {sorted(df_italian['category'].unique())}")

In [ ]:
# Category distribution
category_dist = df_italian["category"].value_counts()

plt.figure(figsize=(12, 6))
category_dist.plot(kind="bar", edgecolor="black", alpha=0.7)
plt.xlabel("Category")
plt.ylabel("Number of Properties")
plt.title("Property Count by Category")
plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Category distribution:")
print(category_dist)

In [ ]:
# Trajectory lengths (properties per trial)
trajectory_lengths = df_italian.groupby(["id", "concept"]).size()

# Plot distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall distribution
axes[0].hist(trajectory_lengths, bins=30, edgecolor="black", alpha=0.7)
axes[0].axvline(
    trajectory_lengths.mean(),
    color="red",
    linestyle="--",
    label=f"Mean: {trajectory_lengths.mean():.1f}",
)
axes[0].axvline(
    trajectory_lengths.median(),
    color="orange",
    linestyle="--",
    label=f"Median: {trajectory_lengths.median():.1f}",
)
axes[0].set_xlabel("Number of Properties per Trial")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Distribution of Trajectory Lengths")
axes[0].legend()
axes[0].grid(alpha=0.3)

# By category (sample for readability)
for cat in ["bird", "mammal", "fruit", "vehicle"]:
    cat_df = df_italian[df_italian["category"] == cat]
    cat_lengths = cat_df.groupby(["id", "concept"]).size()
    axes[1].hist(
        cat_lengths,
        bins=20,
        alpha=0.5,
        label=mappings.categories["italian"].get(cat, cat),
    )

axes[1].set_xlabel("Number of Properties per Trial")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Trajectory Lengths by Category (Sample)")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Trajectory length statistics:")
print(trajectory_lengths.describe())

### 3.2 Text-Level Exploration

In [ ]:
# Most common properties
property_counts = df_italian["property"].value_counts().head(20)

# Visualize
plt.figure(figsize=(12, 6))
property_counts.plot(kind="barh")
plt.xlabel("Frequency")
plt.ylabel("Property")
plt.title("Top 20 Most Common Properties")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("Top 20 most common properties:")
print(property_counts)

In [ ]:
# Property diversity
total_properties = len(df_italian)
unique_properties = df_italian["property"].nunique()
repetition_rate = 1 - (unique_properties / total_properties)

print(f"Total properties: {total_properties}")
print(f"Unique properties: {unique_properties}")
print(f"Repetition rate: {repetition_rate:.2%}")

# Property word length
df_italian["word_length"] = df_italian["property"].str.len()
df_italian["word_count"] = df_italian["property"].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_italian["word_length"].dropna(), bins=30, edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Character Length")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Distribution of Property Character Lengths")
axes[0].grid(alpha=0.3)

axes[1].hist(df_italian["word_count"].dropna(), bins=20, edgecolor="black", alpha=0.7)
axes[1].set_xlabel("Word Count")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Distribution of Property Word Counts")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 3.3 Category Comparisons

In [ ]:
# Trajectory lengths by category
trajectory_by_cat = (
    df_italian.groupby(["category", "id", "concept"])
    .size()
    .reset_index(name="n_properties")
)

plt.figure(figsize=(14, 6))
category_order = sorted(df_italian["category"].unique())
sns.boxplot(
    data=trajectory_by_cat, x="category", y="n_properties", order=category_order
)
plt.xlabel("Category")
plt.ylabel("Number of Properties per Trial")
plt.title("Trajectory Lengths by Category")
plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\nMean trajectory lengths by category:")
print(
    trajectory_by_cat.groupby("category")["n_properties"]
    .mean()
    .sort_values(ascending=False)
)

In [ ]:
# Vocabulary richness by category
vocab_richness = (
    df_italian.groupby(["category", "id"])["property"]
    .nunique()
    .reset_index(name="unique_properties")
)

plt.figure(figsize=(14, 6))
sns.boxplot(
    data=vocab_richness, x="category", y="unique_properties", order=category_order
)
plt.xlabel("Category")
plt.ylabel("Unique Properties per Participant")
plt.title("Vocabulary Richness by Category")
plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\nMean unique properties by category:")
print(
    vocab_richness.groupby("category")["unique_properties"]
    .mean()
    .sort_values(ascending=False)
)

## 4. German Dataset

German students generating descriptive properties for 50 concrete concepts across 10 categories.

In [ ]:
df_german = dfs["German"]
df_german.head()

### 4.1 Basic Statistics

In [ ]:
# Participant and trial counts
n_participants = df_german["id"].nunique()
n_concepts = df_german["concept"].nunique()
n_categories = df_german["category"].nunique()
n_properties = len(df_german)

print(f"Number of participants: {n_participants}")
print(f"Number of concepts: {n_concepts}")
print(f"Number of categories: {n_categories}")
print(f"Total properties generated: {n_properties}")
print(f"\nCategories: {sorted(df_german['category'].unique())}")

In [ ]:
# Category distribution
category_dist = df_german["category"].value_counts()

plt.figure(figsize=(12, 6))
category_dist.plot(kind="bar", edgecolor="black", alpha=0.7)
plt.xlabel("Category")
plt.ylabel("Number of Properties")
plt.title("Property Count by Category")
plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("Category distribution:")
print(category_dist)

In [ ]:
# Trajectory lengths (properties per trial)
trajectory_lengths = df_german.groupby(["id", "concept"]).size()

# Plot distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall distribution
axes[0].hist(trajectory_lengths, bins=30, edgecolor="black", alpha=0.7)
axes[0].axvline(
    trajectory_lengths.mean(),
    color="red",
    linestyle="--",
    label=f"Mean: {trajectory_lengths.mean():.1f}",
)
axes[0].axvline(
    trajectory_lengths.median(),
    color="orange",
    linestyle="--",
    label=f"Median: {trajectory_lengths.median():.1f}",
)
axes[0].set_xlabel("Number of Properties per Trial")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Distribution of Trajectory Lengths")
axes[0].legend()
axes[0].grid(alpha=0.3)

# By category (sample for readability)
for cat in ["bird", "mammal", "fruit", "vehicle"]:
    cat_df = df_german[df_german["category"] == cat]
    cat_lengths = cat_df.groupby(["id", "concept"]).size()
    axes[1].hist(
        cat_lengths,
        bins=20,
        alpha=0.5,
        label=mappings.categories["german"].get(cat, cat),
    )

axes[1].set_xlabel("Number of Properties per Trial")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Trajectory Lengths by Category (Sample)")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Trajectory length statistics:")
print(trajectory_lengths.describe())

### 4.2 Text-Level Exploration

In [ ]:
# Most common properties
property_counts = df_german["property"].value_counts().head(20)

# Visualize
plt.figure(figsize=(12, 6))
property_counts.plot(kind="barh")
plt.xlabel("Frequency")
plt.ylabel("Property")
plt.title("Top 20 Most Common Properties")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("Top 20 most common properties:")
print(property_counts)

In [ ]:
# Property diversity
total_properties = len(df_german)
unique_properties = df_german["property"].nunique()
repetition_rate = 1 - (unique_properties / total_properties)

print(f"Total properties: {total_properties}")
print(f"Unique properties: {unique_properties}")
print(f"Repetition rate: {repetition_rate:.2%}")

# Property word length
df_german["word_length"] = df_german["property"].str.len()
df_german["word_count"] = df_german["property"].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_german["word_length"].dropna(), bins=30, edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Character Length")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Distribution of Property Character Lengths")
axes[0].grid(alpha=0.3)

axes[1].hist(df_german["word_count"].dropna(), bins=20, edgecolor="black", alpha=0.7)
axes[1].set_xlabel("Word Count")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Distribution of Property Word Counts")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 4.3 Category Comparisons

In [ ]:
# Trajectory lengths by category
trajectory_by_cat = (
    df_german.groupby(["category", "id", "concept"])
    .size()
    .reset_index(name="n_properties")
)

plt.figure(figsize=(14, 6))
category_order = sorted(df_german["category"].unique())
sns.boxplot(
    data=trajectory_by_cat, x="category", y="n_properties", order=category_order
)
plt.xlabel("Category")
plt.ylabel("Number of Properties per Trial")
plt.title("Trajectory Lengths by Category")
plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\nMean trajectory lengths by category:")
print(
    trajectory_by_cat.groupby("category")["n_properties"]
    .mean()
    .sort_values(ascending=False)
)

In [ ]:
# Vocabulary richness by category
vocab_richness = (
    df_german.groupby(["category", "id"])["property"]
    .nunique()
    .reset_index(name="unique_properties")
)

plt.figure(figsize=(14, 6))
sns.boxplot(
    data=vocab_richness, x="category", y="unique_properties", order=category_order
)
plt.xlabel("Category")
plt.ylabel("Unique Properties per Participant")
plt.title("Vocabulary Richness by Category")
plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\nMean unique properties by category:")
print(
    vocab_richness.groupby("category")["unique_properties"]
    .mean()
    .sort_values(ascending=False)
)

## 5. Overview: Cross-Dataset Comparison

Comparing patterns across all four datasets to identify similarities and differences.

### 5.1 Dataset Characteristics Summary

In [ ]:
# Create summary table
summary_data = []

for name, df in dfs.items():
    n_participants = df["id"].nunique()
    n_properties = len(df)
    unique_properties = df["property"].nunique()

    # Trajectory lengths
    if name == "Swear Fluency":
        trajectory_lengths = df.groupby(["id", "concept"]).size()
    else:
        trajectory_lengths = df.groupby(["id", "concept"]).size()

    summary_data.append(
        {
            "Dataset": name,
            "Participants": n_participants,
            "Total Properties": n_properties,
            "Unique Properties": unique_properties,
            "Repetition Rate": f"{(1 - unique_properties / n_properties) * 100:.1f}%",
            "Mean Traj Length": f"{trajectory_lengths.mean():.1f}",
            "Median Traj Length": f"{trajectory_lengths.median():.1f}",
            "Std Traj Length": f"{trajectory_lengths.std():.1f}",
        }
    )

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))

In [ ]:
# Visualize key metrics
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Number of participants
participants = [dfs[name]["id"].nunique() for name in datasets.keys()]
axes[0, 0].bar(
    range(len(datasets)), participants, color="steelblue", edgecolor="black", alpha=0.7
)
axes[0, 0].set_xticks(range(len(datasets)))
axes[0, 0].set_xticklabels(datasets.keys(), rotation=45, ha="right")
axes[0, 0].set_ylabel("Count")
axes[0, 0].set_title("Number of Participants by Dataset")
axes[0, 0].grid(alpha=0.3)

# Total properties
total_props = [len(dfs[name]) for name in datasets.keys()]
axes[0, 1].bar(
    range(len(datasets)), total_props, color="coral", edgecolor="black", alpha=0.7
)
axes[0, 1].set_xticks(range(len(datasets)))
axes[0, 1].set_xticklabels(datasets.keys(), rotation=45, ha="right")
axes[0, 1].set_ylabel("Count")
axes[0, 1].set_title("Total Properties Generated by Dataset")
axes[0, 1].grid(alpha=0.3)

# Mean trajectory length
mean_traj = []
for name in datasets.keys():
    df = dfs[name]
    if name == "Swear Fluency":
        traj_len = df.groupby(["id", "concept"]).size().mean()
    else:
        traj_len = df.groupby(["id", "concept"]).size().mean()
    mean_traj.append(traj_len)

axes[1, 0].bar(
    range(len(datasets)), mean_traj, color="lightgreen", edgecolor="black", alpha=0.7
)
axes[1, 0].set_xticks(range(len(datasets)))
axes[1, 0].set_xticklabels(datasets.keys(), rotation=45, ha="right")
axes[1, 0].set_ylabel("Mean Length")
axes[1, 0].set_title("Mean Trajectory Length by Dataset")
axes[1, 0].grid(alpha=0.3)

# Repetition rate
rep_rates = []
for name in datasets.keys():
    df = dfs[name]
    rep_rate = (1 - df["property"].nunique() / len(df)) * 100
    rep_rates.append(rep_rate)

axes[1, 1].bar(
    range(len(datasets)), rep_rates, color="plum", edgecolor="black", alpha=0.7
)
axes[1, 1].set_xticks(range(len(datasets)))
axes[1, 1].set_xticklabels(datasets.keys(), rotation=45, ha="right")
axes[1, 1].set_ylabel("Repetition Rate (\%)")
axes[1, 1].set_title("Property Repetition Rate by Dataset")
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 5.2 Trajectory Length Distributions

In [ ]:
# Compare trajectory length distributions across datasets
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, (name, df) in enumerate(dfs.items()):
    trajectory_lengths = df.groupby(["id", "concept"]).size()

    axes[idx].hist(
        trajectory_lengths, bins=30, edgecolor="black", alpha=0.7, color=f"C{idx}"
    )
    axes[idx].axvline(
        trajectory_lengths.mean(),
        color="red",
        linestyle="--",
        label=f"Mean: {trajectory_lengths.mean():.1f}",
    )
    axes[idx].axvline(
        trajectory_lengths.median(),
        color="orange",
        linestyle="--",
        label=f"Median: {trajectory_lengths.median():.1f}",
    )
    axes[idx].set_xlabel("Number of Properties per Trial")
    axes[idx].set_ylabel("Frequency")
    axes[idx].set_title(f"{name} Dataset")
    axes[idx].legend()
    axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 5.3 Language Comparison: Italian vs German

In [ ]:
# Compare Italian and German datasets (same task structure)
print("Cross-Linguistic Comparison: Italian vs German\n")

# Trajectory lengths
italian_traj = dfs["Italian"].groupby(["id", "concept"]).size()
german_traj = dfs["German"].groupby(["id", "concept"]).size()

print("Trajectory Length Comparison:")
print(
    f"Italian - Mean: {italian_traj.mean():.2f}, Median: {italian_traj.median():.1f}, Std: {italian_traj.std():.2f}"
)
print(
    f"German  - Mean: {german_traj.mean():.2f}, Median: {german_traj.median():.1f}, Std: {german_traj.std():.2f}"
)

# Compare by category
italian_by_cat = (
    dfs["Italian"]
    .groupby(["category", "id", "concept"])
    .size()
    .reset_index(name="n_properties")
)
german_by_cat = (
    dfs["German"]
    .groupby(["category", "id", "concept"])
    .size()
    .reset_index(name="n_properties")
)

# Merge for comparison
italian_cat_mean = italian_by_cat.groupby("category")["n_properties"].mean()
german_cat_mean = german_by_cat.groupby("category")["n_properties"].mean()

comparison_df = pd.DataFrame(
    {
        "Italian": italian_cat_mean,
        "German": german_cat_mean,
        "Difference": italian_cat_mean - german_cat_mean,
    }
)

print("\nMean Trajectory Length by Category:")
print(comparison_df.round(2))

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Side-by-side bar chart
x = np.arange(len(comparison_df))
width = 0.35

axes[0].bar(
    x - width / 2,
    comparison_df["Italian"],
    width,
    label="Italian",
    alpha=0.8,
    edgecolor="black",
)
axes[0].bar(
    x + width / 2,
    comparison_df["German"],
    width,
    label="German",
    alpha=0.8,
    edgecolor="black",
)
axes[0].set_xlabel("Category")
axes[0].set_ylabel("Mean Trajectory Length")
axes[0].set_title("Mean Trajectory Length by Category: Italian vs German")
axes[0].set_xticks(x)
axes[0].set_xticklabels(comparison_df.index, rotation=45, ha="right")
axes[0].legend()
axes[0].grid(alpha=0.3)

# Difference plot
colors = ["green" if x > 0 else "red" for x in comparison_df["Difference"]]
axes[1].bar(
    range(len(comparison_df)),
    comparison_df["Difference"],
    color=colors,
    alpha=0.7,
    edgecolor="black",
)
axes[1].axhline(0, color="black", linestyle="-", linewidth=0.8)
axes[1].set_xlabel("Category")
axes[1].set_ylabel("Difference (Italian - German)")
axes[1].set_title("Trajectory Length Difference by Category")
axes[1].set_xticks(range(len(comparison_df)))
axes[1].set_xticklabels(comparison_df.index, rotation=45, ha="right")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### 5.4 Task Type Comparison

In [ ]:
# Compare property listing vs verbal fluency
property_listing = ["Neurodegenerative", "Italian", "German"]
verbal_fluency = ["Swear Fluency"]

# Calculate statistics for each task type
pl_stats = []
for name in property_listing:
    traj = dfs[name].groupby(["id", "concept"]).size()
    pl_stats.append(traj.mean())

vf_stats = []
for name in verbal_fluency:
    traj = dfs[name].groupby(["id", "concept"]).size()
    vf_stats.append(traj.mean())

print(
    f"Property Listing Tasks - Mean trajectory length: {np.mean(pl_stats):.2f} (±{np.std(pl_stats):.2f})"
)
print(f"Verbal Fluency Tasks   - Mean trajectory length: {np.mean(vf_stats):.2f}")

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))

all_traj_data = []
all_labels = []
all_colors = []

for name in datasets.keys():
    traj = dfs[name].groupby(["id", "concept"]).size()
    all_traj_data.append(traj)
    all_labels.append(name)
    if name in property_listing:
        all_colors.append("steelblue")
    else:
        all_colors.append("coral")

bp = ax.boxplot(all_traj_data, tick_labels=all_labels, patch_artist=True)
for patch, color in zip(bp["boxes"], all_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_ylabel("Trajectory Length (Properties per Trial)")
ax.set_title(
    "Trajectory Length Distribution by Dataset\n(Blue = Property Listing, Orange = Verbal Fluency)"
)
ax.grid(alpha=0.3, axis="y")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

### 5.5 Key Findings Summary

In [ ]:
print("\n" + "=" * 80)
print("KEY FINDINGS SUMMARY")
print("=" * 80)

print("\n1. DATASET SCALE:")
for name, df in dfs.items():
    print(f"   - {name}: {df['id'].nunique()} participants, {len(df)} properties")

print("\n2. TRAJECTORY CHARACTERISTICS:")
for name, df in dfs.items():
    traj = df.groupby(["id", "concept"]).size()
    print(
        f"   - {name}: Mean={traj.mean():.1f}, Median={traj.median():.1f}, Range=[{traj.min()}-{traj.max()}]"
    )

print("\n3. VOCABULARY DIVERSITY:")
for name, df in dfs.items():
    rep_rate = (1 - df["property"].nunique() / len(df)) * 100
    print(
        f"   - {name}: {df['property'].nunique()} unique properties, {rep_rate:.1f}% repetition rate"
    )

print("\n4. CROSS-LINGUISTIC PATTERNS (Italian vs German):")
italian_mean = dfs["Italian"].groupby(["id", "concept"]).size().mean()
german_mean = dfs["German"].groupby(["id", "concept"]).size().mean()
print(
    f"   - Similar trajectory lengths: Italian={italian_mean:.1f}, German={german_mean:.1f}"
)
print(
    f"   - Both have same 10 semantic categories with comparable property generation patterns"
)

print("\n5. CLINICAL vs HEALTHY (Neurodegenerative):")
if "category" in dfs["Neurodegenerative"].columns:
    for cat in ["CN", "PD", "DF"]:
        cat_df = dfs["Neurodegenerative"][dfs["Neurodegenerative"]["category"] == cat]
        traj = cat_df.groupby(["id", "concept"]).size()
        cat_label = mappings.categories["parkinson"][cat]
        print(f"   - {cat_label}: Mean={traj.mean():.1f} properties/trial")